# 03_ML_Sales_Prediction

## Rossmann Sales Forecasting — Random Forest Sales Model

This notebook develops the primary **tree-based Sales forecasting model** for Project 6.

### Objective
Predict daily Rossmann store Sales using store, calendar, promotion, holiday, assortment, and competition-related features.

### Workflow
1. Prepare and validate the modelling dataset
2. Define features and a chronological validation strategy
3. Build a reproducible scikit-learn preprocessing pipeline
4. Train a Random Forest Regressor
5. Evaluate the model using MAE, RMSE, and R²
6. Inspect feature importance
7. Estimate an approximate prediction interval
8. Serialize the trained pipeline with a timestamp
9. Generate predictions for the supplied Rossmann test dataset
10. Save the final Sales submission CSV

> **Validation note:** July 2015 is used as a historical validation period. The official Rossmann test set does not contain Sales, so its predictions are generated only after validation is complete.


## 1. Dataset Preparation for Machine Learning

The modelling dataset combines the transaction-level training data with store-level attributes. Calendar features created during EDA are reused here so the model can learn time and seasonal patterns.

In [ ]:
import pandas as pd

# Load Rossmann datasets using paths relative to the project structure.
# This notebook is stored in the "notebooks" folder.
train = pd.read_csv('../train.csv')
store = pd.read_csv('../store.csv')
test = pd.read_csv('../test.csv')
sample_submission = pd.read_csv('../sample_submission.csv')

print("Train shape:", train.shape)
print("Store shape:", store.shape)
print("Test shape:", test.shape)
print("Sample submission shape:", sample_submission.shape)


Train shape: (1017209, 15)
Store shape: (1115, 10)
Test shape: (41088, 8)
Sample submission shape: (41088, 2)


### Test-set structure and submission template

In [ ]:
print("TEST SHAPE:", test.shape)
print("TEST COLUMNS:")
print(test.columns.tolist())

print("\nTEST DATA TYPES:")
print(test.dtypes)

print("\nTEST MISSING VALUES:")
print(test.isnull().sum())

print("\nSAMPLE SUBMISSION SHAPE:", sample_submission.shape)
print("SAMPLE SUBMISSION COLUMNS:")
print(sample_submission.columns.tolist())

print("\nSAMPLE SUBMISSION FIRST 5 ROWS:")
print(sample_submission.head())


TEST SHAPE: (41088, 8)
TEST COLUMNS:
['Id', 'Store', 'DayOfWeek', 'Date', 'Open', 'Promo', 'StateHoliday', 'SchoolHoliday']

TEST DATA TYPES:
Id                 int64
Store              int64
DayOfWeek          int64
Date              object
Open             float64
Promo              int64
StateHoliday      object
SchoolHoliday      int64
dtype: object

TEST MISSING VALUES:
Id                0
Store             0
DayOfWeek         0
Date              0
Open             11
Promo             0
StateHoliday      0
SchoolHoliday     0
dtype: int64

SAMPLE SUBMISSION SHAPE: (41088, 2)
SAMPLE SUBMISSION COLUMNS:
['Id', 'Sales']

SAMPLE SUBMISSION FIRST 5 ROWS:
   Id  Sales
0   1      0
1   2      0
2   3      0
3   4      0
4   5      0


### Date handling, data-type standardization, and store merge

In [ ]:
# Convert dates
train['Date'] = pd.to_datetime(train['Date'])
test['Date'] = pd.to_datetime(test['Date'])

# Standardize StateHoliday so numeric 0 and string '0' are treated consistently.
train['StateHoliday'] = train['StateHoliday'].astype(str)
test['StateHoliday'] = test['StateHoliday'].astype(str)

# Calendar features for TRAIN
train['Year'] = train['Date'].dt.year
train['Month'] = train['Date'].dt.month
train['Day'] = train['Date'].dt.day
train['WeekOfYear'] = train['Date'].dt.isocalendar().week.astype(int)
train['IsWeekend'] = train['DayOfWeek'].isin([6, 7]).astype(int)
train['YearMonth'] = train['Date'].dt.to_period('M').astype(str)

# Calendar features for TEST
test['Year'] = test['Date'].dt.year
test['Month'] = test['Date'].dt.month
test['Day'] = test['Date'].dt.day
test['WeekOfYear'] = test['Date'].dt.isocalendar().week.astype(int)
test['IsWeekend'] = test['DayOfWeek'].isin([6, 7]).astype(int)
test['YearMonth'] = test['Date'].dt.to_period('M').astype(str)

# Merge transaction data with store attributes.
merged = train.merge(store, on='Store', how='left')

# Working dataframe for modelling
df = merged.copy()

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Merged shape:", merged.shape)
print("DF shape:", df.shape)

print("\nMissing StoreType after merge:", df['StoreType'].isnull().sum())
print("Missing Assortment after merge:", df['Assortment'].isnull().sum())


Train shape: (1017209, 15)
Test shape: (41088, 14)
Merged shape: (1017209, 24)
DF shape: (1017209, 24)

Missing StoreType after merge: 0
Missing Assortment after merge: 0


### Modelling-data decisions

- `Sales` is the target variable.
- `Customers` is **not included** in the initial Sales model because future customer counts are not available for the official test rows; using them would create a deployment-time availability problem.
- The feature set contains store, calendar, promotion, holiday, assortment, and competition information.
- Missing numerical values are handled inside the modelling pipeline rather than by manually altering the raw dataset.


In [ ]:
# Features for the Sales Prediction model
features = [
    'Store',
    'DayOfWeek',
    'Open',
    'Promo',
    'StateHoliday',
    'SchoolHoliday',
    'Year',
    'Month',
    'Day',
    'WeekOfYear',
    'IsWeekend',
    'StoreType',
    'Assortment',
    'CompetitionDistance',
    'CompetitionOpenSinceMonth',
    'CompetitionOpenSinceYear',
    'Promo2',
    'Promo2SinceWeek',
    'Promo2SinceYear',
    'PromoInterval'
]

X = df[features]
y = df['Sales']

print("Number of features:", len(features))
print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nFeatures:")
print(features)

Number of features: 20
X shape: (1017209, 20)
y shape: (1017209,)

Features:
['Store', 'DayOfWeek', 'Open', 'Promo', 'StateHoliday', 'SchoolHoliday', 'Year', 'Month', 'Day', 'WeekOfYear', 'IsWeekend', 'StoreType', 'Assortment', 'CompetitionDistance', 'CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear', 'Promo2', 'Promo2SinceWeek', 'Promo2SinceYear', 'PromoInterval']


## 2. Chronological Train–Validation Split

A time-based split is used instead of a random split because this is a forecasting problem. The model is trained on historical observations and validated on the later July 2015 period.

- **Training:** 2013-01-01 to 2015-06-30
- **Validation:** 2015-07-01 to 2015-07-31

This reduces the risk of evaluating the model using future information from randomly mixed dates.


In [ ]:
# Time-based train-validation split

train_ml = df[df['Date'] < '2015-07-01'].copy()
test_ml = df[df['Date'] >= '2015-07-01'].copy()

X_train = train_ml[features]
y_train = train_ml['Sales']

X_test = test_ml[features]
y_test = test_ml['Sales']

print("Training period:", train_ml['Date'].min(), "to", train_ml['Date'].max())
print("Validation period:", test_ml['Date'].min(), "to", test_ml['Date'].max())

print("\nTraining rows:", len(train_ml))
print("Validation rows:", len(test_ml))

print("\nX_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

Training period: 2013-01-01 00:00:00 to 2015-06-30 00:00:00
Validation period: 2015-07-01 00:00:00 to 2015-07-31 00:00:00

Training rows: 982644
Validation rows: 34565

X_train shape: (982644, 20)
X_test shape: (34565, 20)


## 3. Preprocessing Pipeline

The pipeline separates numerical and categorical features.

- Numerical features → median imputation
- Categorical features → most-frequent imputation + one-hot encoding
- Unknown categories at prediction time are ignored by `OneHotEncoder(handle_unknown='ignore')`
- Random Forest does **not** require feature scaling, so no scaler is applied here.

Keeping preprocessing inside the same scikit-learn pipeline ensures that training and inference use the same transformations.


In [ ]:
# Separate numerical and categorical features

numeric_features = [
    'Store',
    'DayOfWeek',
    'Open',
    'Promo',
    'SchoolHoliday',
    'Year',
    'Month',
    'Day',
    'WeekOfYear',
    'IsWeekend',
    'CompetitionDistance',
    'CompetitionOpenSinceMonth',
    'CompetitionOpenSinceYear',
    'Promo2',
    'Promo2SinceWeek',
    'Promo2SinceYear'
]

categorical_features = [
    'StateHoliday',
    'StoreType',
    'Assortment',
    'PromoInterval'
]

print("Number of numerical features:", len(numeric_features))
print("Number of categorical features:", len(categorical_features))

print("\nNumerical features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

Number of numerical features: 16
Number of categorical features: 4

Numerical features:
['Store', 'DayOfWeek', 'Open', 'Promo', 'SchoolHoliday', 'Year', 'Month', 'Day', 'WeekOfYear', 'IsWeekend', 'CompetitionDistance', 'CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear', 'Promo2', 'Promo2SinceWeek', 'Promo2SinceYear']

Categorical features:
['StateHoliday', 'StoreType', 'Assortment', 'PromoInterval']


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

# Numerical preprocessing
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])

# Categorical preprocessing
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Combine both preprocessing pipelines
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

print("Preprocessing pipeline created successfully.")
print("\nNumerical columns:", len(numeric_features))
print("Categorical columns:", len(categorical_features))

Preprocessing pipeline created successfully.

Numerical columns: 16
Categorical columns: 4


## 4. Random Forest Model

### Why Random Forest?

Random Forest is a strong baseline for this problem because it can capture nonlinear relationships and interactions among store, calendar, promotion, holiday, and competition features. It also works naturally with mixed numerical and one-hot encoded categorical inputs.

### Model configuration

- `n_estimators = 100`
- `max_depth = 20`
- `min_samples_leaf = 2`
- `random_state = 42`
- `n_jobs = -1`

The complete preprocessing + model workflow is kept in one pipeline.


In [ ]:
from sklearn.ensemble import RandomForestRegressor

# Random Forest model
rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=20,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

# Complete ML pipeline
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', rf_model)
])

print("Random Forest pipeline created successfully.")
print("\nModel configuration:")
print(rf_model)

Random Forest pipeline created successfully.

Model configuration:
RandomForestRegressor(max_depth=20, min_samples_leaf=2, n_jobs=-1,
                      random_state=42)


### Loss function and evaluation strategy

The Random Forest uses **squared error (MSE)** as its default split criterion. Squared error penalizes larger errors more strongly, which is useful when large sales misses have greater operational impact.

For business evaluation, three complementary validation metrics are reported:

- **MAE:** average absolute forecasting error
- **RMSE:** penalizes larger errors more heavily
- **R²:** proportion of variance explained by the model

These metrics should be interpreted together rather than relying on a single score.


## 5. Model Training — Existing Completed Run

> **Important:** The following training cell contains the completed validated run. The original training run took approximately 10 minutes. Do not rerun it unnecessarily merely to reproduce the notebook layout.


In [ ]:
import time

print("Starting Random Forest training...")
start_time = time.time()

rf_pipeline.fit(X_train, y_train)

end_time = time.time()

print("\nRandom Forest training completed successfully.")
print("Training time: {:.2f} minutes".format((end_time - start_time) / 60))

Starting Random Forest training...

Random Forest training completed successfully.
Training time: 10.08 minutes


## 6. Validation and Evaluation

The July 2015 historical validation set is used to evaluate the trained model.

In [ ]:
import time

print("Starting validation predictions...")
start_time = time.time()

y_pred = rf_pipeline.predict(X_test)

end_time = time.time()

print("\nValidation predictions completed successfully.")
print("Prediction time: {:.2f} minutes".format((end_time - start_time) / 60))

print("\nNumber of predictions:", len(y_pred))
print("First 10 predictions:")
print(y_pred[:10])

Starting validation predictions...

Validation predictions completed successfully.
Prediction time: 0.01 minutes

Number of predictions: 34565
First 10 predictions:
[ 5663.06208719  6913.49660891  9125.33534393 12214.92955219
  6397.53322719  6590.71613853 12140.4446014   7517.44830925
  9714.22217769  6747.74084804]


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Calculate evaluation metrics
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("Random Forest Validation Results")
print("--------------------------------")
print("MAE :", round(mae, 2))
print("RMSE:", round(rmse, 2))
print("R²  :", round(r2, 4))

Random Forest Validation Results
--------------------------------
MAE : 809.24
RMSE: 1218.28
R²  : 0.8859


### Validation interpretation

The completed Random Forest achieved approximately:

- **MAE = 809.24**
- **RMSE = 1,218.28**
- **R² = 0.8859**

These scores indicate strong predictive performance on the selected historical validation period. They are **validation results**, not a score on the official hidden Rossmann test set because the test set has no observed Sales values.


## 7. Feature Importance

Feature importance describes how much the trained Random Forest relied on each transformed feature for reducing prediction error. It should not be interpreted as causal impact.

In [ ]:
# Get the trained Random Forest model
rf_trained_model = rf_pipeline.named_steps['model']

# Get feature names after preprocessing
feature_names = rf_pipeline.named_steps['preprocessor'].get_feature_names_out()

# Get feature importance
importances = rf_trained_model.feature_importances_

# Create feature importance table
feature_importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
})

# Sort from highest to lowest
feature_importance_df = feature_importance_df.sort_values(
    by='Importance',
    ascending=False
)

print("Top 20 Important Features:")
print(feature_importance_df.head(20).to_string(index=False))

Top 20 Important Features:
                            Feature  Importance
                          num__Open    0.508711
           num__CompetitionDistance    0.092245
                         num__Promo    0.080344
                         num__Store    0.070218
      num__CompetitionOpenSinceYear    0.041116
     num__CompetitionOpenSinceMonth    0.033846
                     num__DayOfWeek    0.029371
               num__Promo2SinceWeek    0.018019
                    num__WeekOfYear    0.016021
                   cat__StoreType_b    0.014949
                           num__Day    0.014714
               num__Promo2SinceYear    0.012751
                        num__Promo2    0.010943
                  cat__Assortment_c    0.008847
                   cat__StoreType_a    0.007290
                         num__Month    0.006073
                  cat__Assortment_a    0.006021
                   cat__StoreType_c    0.005655
                          num__Year    0.004828
cat__PromoInt

### Feature importance interpretation

The strongest model driver is `Open`, which is expected because closed stores have zero Sales in the training data. Other influential variables include competition distance, promotion, store identity, competition opening information, day of week, and promotional timing features.

A feature's importance indicates model reliance; it does **not** prove that the feature causes Sales to change.


## 8. Approximate Prediction Interval

The ensemble's tree-level prediction spread is used to construct an approximate 95% model-based prediction interval.

In [ ]:
import numpy as np

print("Calculating prediction confidence intervals...")

# Transform validation data using the trained preprocessing pipeline
X_test_transformed = rf_pipeline.named_steps['preprocessor'].transform(X_test)

# Get the individual trees from the trained Random Forest
trees = rf_pipeline.named_steps['model'].estimators_

# Generate predictions from each individual tree
tree_predictions = np.array([
    tree.predict(X_test_transformed)
    for tree in trees
])

# Calculate prediction statistics
prediction_mean = tree_predictions.mean(axis=0)
prediction_std = tree_predictions.std(axis=0)

# Approximate 95% prediction interval
lower_bound = prediction_mean - 1.96 * prediction_std
upper_bound = prediction_mean + 1.96 * prediction_std

# Prevent negative Sales predictions
lower_bound = np.maximum(lower_bound, 0)
upper_bound = np.maximum(upper_bound, 0)

# Create confidence interval dataframe
confidence_df = pd.DataFrame({
    'Actual_Sales': y_test.values,
    'Predicted_Sales': prediction_mean,
    'Lower_Bound': lower_bound,
    'Upper_Bound': upper_bound
})

print("Confidence interval calculation completed.")
print("\nFirst 10 predictions with intervals:")
print(confidence_df.head(10).round(2).to_string(index=False))

Calculating prediction confidence intervals...
Confidence interval calculation completed.

First 10 predictions with intervals:
 Actual_Sales  Predicted_Sales  Lower_Bound  Upper_Bound
         5263          5663.06      4515.41      6810.72
         6064          6913.50      4041.98      9785.01
         8314          9125.34      6923.98     11326.69
        13995         12214.93      8285.04     16144.82
         4822          6397.53      4794.01      8001.06
         5651          6590.72      4520.58      8660.85
        15344         12140.44      7496.79     16784.10
         8492          7517.45      5491.41      9543.48
         8565          9714.22      6030.69     13397.75
         7185          6747.74      5278.02      8217.46


### Prediction interval caveat

The interval above is an **approximate model-based prediction interval** derived from variation among individual Random Forest trees. It is useful for communicating uncertainty, but it is **not a formally calibrated statistical confidence interval**.

The lower bound is clipped at zero because Sales cannot be negative.


## 9. Model Serialization

The complete trained preprocessing + Random Forest pipeline is serialized with a timestamp so that the validated model can be reused for inference.

In [ ]:
import joblib
from datetime import datetime
import os

# Create a timestamp for versioned model artifacts.
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Save the complete trained pipeline inside the project models folder.
model_dir = "../models"
os.makedirs(model_dir, exist_ok=True)

model_filename = f"{model_dir}/rossmann_random_forest_{timestamp}.joblib"

joblib.dump(rf_pipeline, model_filename)

print("Model saved successfully!")
print("File:", model_filename)


Model saved successfully!
File: ../models/rossmann_random_forest_20260909_084440.joblib


In [ ]:
import os

print("Saved model exists:", os.path.exists(model_filename))

if os.path.exists(model_filename):
    size_mb = os.path.getsize(model_filename) / (1024 * 1024)
    print("Model file size: {:.2f} MB".format(size_mb))


Saved model exists: True
Model file size: 501.92 MB


### Validated saved artifact

The completed project run produced the timestamped Sales model artifact:

`rossmann_random_forest_20260909_084440.joblib`

The saved pipeline contains both preprocessing and the Random Forest model, so the same transformations are applied during inference.


## 10. Final Test Preparation and Sales Prediction

The official Rossmann test data contains no Sales target. The trained pipeline is applied to the test rows after adding the same store and calendar features used during training.

In [ ]:
# Create a copy so the original test data remains unchanged
test_final = test.copy()

# Merge store information
test_final = test_final.merge(
    store,
    on='Store',
    how='left'
)

# Feature engineering
test_final['Year'] = test_final['Date'].dt.year
test_final['Month'] = test_final['Date'].dt.month
test_final['Day'] = test_final['Date'].dt.day
test_final['WeekOfYear'] = test_final['Date'].dt.isocalendar().week.astype(int)
test_final['IsWeekend'] = test_final['DayOfWeek'].isin([6, 7]).astype(int)
test_final['YearMonth'] = test_final['Date'].dt.to_period('M').astype(str)

# Standardize StateHoliday
test_final['StateHoliday'] = test_final['StateHoliday'].astype(str)

print("Test final shape:", test_final.shape)

print("\nMissing StoreType:",
      test_final['StoreType'].isnull().sum())

print("Missing Assortment:",
      test_final['Assortment'].isnull().sum())

print("\nTest final columns:")
print(test_final.columns.tolist())

Test final shape: (41088, 23)

Missing StoreType: 0
Missing Assortment: 0

Test final columns:
['Id', 'Store', 'DayOfWeek', 'Date', 'Open', 'Promo', 'StateHoliday', 'SchoolHoliday', 'Year', 'Month', 'Day', 'WeekOfYear', 'IsWeekend', 'YearMonth', 'StoreType', 'Assortment', 'CompetitionDistance', 'CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear', 'Promo2', 'Promo2SinceWeek', 'Promo2SinceYear', 'PromoInterval']


In [ ]:
print("Starting final Sales prediction...")

# Select exactly the same 20 features used during training
X_final_test = test_final[features]

# Generate Sales predictions
final_sales_predictions = rf_pipeline.predict(X_final_test)

print("\nFinal Sales predictions completed successfully.")
print("Number of predictions:", len(final_sales_predictions))

print("\nFirst 10 predictions:")
print(final_sales_predictions[:10])

Starting final Sales prediction...

Final Sales predictions completed successfully.
Number of predictions: 41088

First 10 predictions:
[4964.94166091 7957.84167127 8350.69918843 6556.8465258  7699.62385178
 6493.79841401 8178.52525046 7791.35663191 5560.77204572 6318.59967692]


In [ ]:
# Create final submission dataframe
submission = sample_submission.copy()

# Replace sample Sales values with our model predictions
submission['Sales'] = final_sales_predictions

# Make sure Sales predictions are not negative
submission['Sales'] = submission['Sales'].clip(lower=0)

print("Submission shape:", submission.shape)
print("\nSubmission columns:")
print(submission.columns.tolist())

print("\nFirst 10 submission rows:")
print(submission.head(10))

Submission shape: (41088, 2)

Submission columns:
['Id', 'Sales']

First 10 submission rows:
   Id        Sales
0   1  4964.941661
1   2  7957.841671
2   3  8350.699188
3   4  6556.846526
4   5  7699.623852
5   6  6493.798414
6   7  8178.525250
7   8  7791.356632
8   9  5560.772046
9  10  6318.599677


In [ ]:
# Save final Sales predictions in the project root.
submission_file = '../rossmann_sales_predictions.csv'

submission.to_csv(submission_file, index=False)

print("Final submission saved successfully!")
print("File:", submission_file)

# Verify the saved file
saved_submission = pd.read_csv(submission_file)

print("\nSaved file shape:", saved_submission.shape)
print("Saved file columns:", saved_submission.columns.tolist())
print("\nFirst 5 rows:")
print(saved_submission.head())


Final submission saved successfully!
File: ../rossmann_sales_predictions.csv

Saved file shape: (41088, 2)
Saved file columns: ['Id', 'Sales']

First 5 rows:
   Id        Sales
0   1  4964.941661
1   2  7957.841671
2   3  8350.699188
3   4  6556.846526
4   5  7699.623852


## 11. Final Sales Model Conclusion

The completed Random Forest Sales model achieved approximately **MAE 809.24, RMSE 1,218.28, and R² 0.8859** on the July 2015 chronological validation period.

The model uses a reproducible scikit-learn pipeline for missing-value handling and categorical encoding, was serialized as a timestamped artifact, and generated **41,088 Sales predictions** for the official Rossmann test set.

### Key modelling decisions
- Chronological validation rather than random splitting
- Customers excluded from the initial Sales model because future customer counts are unavailable at prediction time
- Median imputation for numerical missing values
- Most-frequent imputation + one-hot encoding for categorical features
- Random Forest for nonlinear relationships and interactions
- Squared-error objective with MAE/RMSE/R² reported for evaluation
- Approximate ensemble-based prediction interval for uncertainty communication

### Handoff
The validated Sales model and its test predictions are subsequently used by the MLflow, customer-prediction, and deployment components of Project 6.
